# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [1]:
!nvidia-smi # Take a look at the GPU

Sat Mar 28 10:08:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   24C    P8              1W /  250W |    3782MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install numpy
%pip install torch torchvision
%pip install wandb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any
from utils import train, validate, fit

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Data Loading

In [4]:
BATCH_SIZE = 256 # Same batch size throughout notebook

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [7]:
# Hyperparameters
NUM_EPOCHS = 5
LEARNING_RATE = 1e-2

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SGD + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_a = fit(
    model=model_a, 
    optimizer=optimizer_a,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=test_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oscar-engelmark (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/5 | train loss 2.2999, train acc 10.62% | val loss 2.2958, val acc 11.00%
Epoch 2/5 | train loss 2.2890, train acc 12.53% | val loss 2.2784, val acc 15.03%
Epoch 3/5 | train loss 2.2494, train acc 18.40% | val loss 2.1961, val acc 21.74%
Epoch 4/5 | train loss 2.1123, train acc 25.86% | val loss 2.0354, val acc 28.88%
Epoch 5/5 | train loss 1.9690, train acc 30.28% | val loss 1.9188, val acc 32.64%


Training Accuracy,▁▂▄▆█
Training Loss,██▇▄▁
Validation Accuracy,▁▂▄▇█
Validation Loss,██▆▃▁
Training Accuracy,30.284
Training Loss,1.96904
Validation Accuracy,32.64
Validation Loss,1.91878



Restored best weights (val acc 32.64%)


## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_b = fit(
    model=model_b, 
    optimizer=optimizer_b,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=test_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS
)

Epoch 1/5 | train loss 1.5275, train acc 44.40% | val loss 1.2378, val acc 55.03%
Epoch 2/5 | train loss 1.1225, train acc 59.84% | val loss 1.0152, val acc 64.51%
Epoch 3/5 | train loss 0.9376, train acc 66.90% | val loss 0.9387, val acc 66.94%
Epoch 4/5 | train loss 0.8138, train acc 71.31% | val loss 0.8643, val acc 69.14%
Epoch 5/5 | train loss 0.7213, train acc 74.62% | val loss 0.8234, val acc 71.31%


Training Accuracy,▁▅▆▇█
Training Loss,█▄▃▂▁
Validation Accuracy,▁▅▆▇█
Validation Loss,█▄▃▂▁
Training Accuracy,74.618
Training Loss,0.7213
Validation Accuracy,71.31
Validation Loss,0.82339



Restored best weights (val acc 71.31%)


## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [9]:
# Hyperparameters
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + Tanh",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_c = fit(
    model=model_c, 
    optimizer=optimizer_c,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=test_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS
)

Epoch 1/5 | train loss 1.4007, train acc 49.95% | val loss 1.1232, val acc 59.87%
Epoch 2/5 | train loss 1.0162, train acc 64.11% | val loss 0.9589, val acc 66.09%
Epoch 3/5 | train loss 0.8458, train acc 70.40% | val loss 0.8763, val acc 69.52%
Epoch 4/5 | train loss 0.7180, train acc 75.23% | val loss 0.8112, val acc 72.15%
Epoch 5/5 | train loss 0.6256, train acc 78.21% | val loss 0.7944, val acc 72.49%


Training Accuracy,▁▅▆▇█
Training Loss,█▅▃▂▁
Validation Accuracy,▁▄▆██
Validation Loss,█▅▃▁▁
Training Accuracy,78.206
Training Loss,0.62558
Validation Accuracy,72.49
Validation Loss,0.7944



Restored best weights (val acc 72.49%)
